# Transformer
编码器：多头自注意力层（Multi-Head Self-Attention Layer）和位置前馈网络（Position-wise Feed-Forward Network）。每个子层的输出都经过了残差连接（Add）与层归一化（Norm）处理。<br>
解码器：带掩码的多头自注意力层（Masked Multi-Head Self-Attention Layer）、交叉注意力层（Cross-Attention Layer）和位置前馈网络（Position-wise Feed-Forward Network）。同样，解码器的每个子层也都采用了残差连接和层归一化。<br>

位置前馈网络：由两次线性变换和一个激活函数组成的全连接网络
$$ \text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2 $$
位置编码
$$ input_\text{embedding} = token_\text{embedding} + positional_\text{encoding} $$
$$ PE_{(\text{pos}, 2i)} = \sin\left(\frac{\text{pos}}{10000^{\frac{2i}{d_{\text{embedding}}}}}\right) \ PE_{(\text{pos}, 2i+1)} = \cos\left(\frac{\text{pos}}{10000^{\frac{2i}{d_{\text{embedding}}}}}\right) $$
层归一化
$$ y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta $$

## Encoder
自注意力层+归一化+前馈网络+归一化
$$ \text{EncoderLayer}(x) = \text{Norm}(\text{FFN}(\text{Norm}(\text{SelfAttention}(x) + x)) + \text{Norm}(\text{SelfAttention}(x) + x)) $$
## Decoder
掩码自注意力层+归一化+交叉注意力层+归一化+前馈网络+归一化
$$ \text{DecoderLayer}(x, enc\_output) = \text{Norm}(\text{FFN}(\text{Norm}(\text{CrossAttention}(\text{Norm}(\text{MaskedSelfAttention}(x) + x), enc\_output) + \text{Norm}(\text{MaskedSelfAttention}(x) + x))) + \text{Norm}(\text{CrossAttention}(\text{Norm}(\text{MaskedSelfAttention}(x) + x), enc\_output) + \text{Norm}(\text{MaskedSelfAttention}(x) + x))) $$

In [ ]:
import torch.nn as nn
import torch
import math
class PositionalEncoding(nn.Module):#位置编码
    def __init__(self,dim,max_seq_len=5000):
        super().__init__()
        pe = torch.zeros(max_seq_len,dim)
        position = torch.arange(0,max_seq_len).unsqueeze(1)# [max_seq_len,1]
        div = torch.exp(torch.arange(0,dim,2)*(-math.log(10000)/dim))# [dim/2]
        pe[:,0::2] = torch.sin(position*div)# [max_seq_len,dim/2],偶数
        pe[:,1::2] = torch.cos(position*div)# [max_seq_len,dim/2],奇数
        pe = pe.unsqueeze(0)# [1,max_seq_len,dim]，便于广播
        self.register_buffer('pe',pe)
    def forward(self,x):
        x = x + self.pe[:,:x.size(1),:]# x: [batch_size,seq_len,dim]
        return x
    
class MultiHeadAttention(nn.Module):##子注意力机制
    def __init__(self,dim,n_heads,dropout=0.1):
        super().__init__()
        self.dim = dim
        self.n_dim = dim // n_heads
        self.n_heads = n_heads
        self.q_linear = nn.Linear(dim,dim)
        self.k_linear = nn.Linear(dim,dim)
        self.v_linear = nn.Linear(dim,dim)
        self.fc_out = nn.Linear(dim,dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self,query,key,value,mask=None):
        batch_size = query.size(0)
        
        q = self.q_linear(query)# [batch_size,seq_len,dim]
        k = self.k_linear(key)
        v = self.v_linear(value)
        q = q.view(batch_size,-1,self.n_heads,self.n_dim).transpose(1,2)# [batch_size,n_heads,seq_len,n_dim]
        k = k.view(batch_size,-1,self.n_heads,self.n_dim).transpose(1,2)
        v = v.view(batch_size,-1,self.n_heads,self.n_dim).transpose(1,2)
        scores = torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.n_dim)# [batch_size,n_heads,seq_len,seq_len]
        if mask is not None:#掩码的遮蔽
            scores = scores.masked_fill(mask==0,float('-inf'))
        attention = torch.softmax(scores,dim=-1)
        if self.dropout is not None:
            attention = self.dropout(attention)
        context = torch.matmul(attention,v)# [batch_size,n_heads,seq_len,n_dim]
        context = context.transpose(1,2).contiguous().view(batch_size,-1,self.dim)# [batch_size,seq_len,dim]
        output = self.fc_out(context)# [batch_size,seq_len,dim]
        return output
    
class FeedForward(nn.Module):#位置前馈网络
    def __init__(self,dim,hidden_dim,dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(dim,hidden_dim)
        self.fc2 = nn.Linear(hidden_dim,dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    def forward(self,x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class layerNorm(nn.Module):#层归一化
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
    def forward(self,x):
        mean = x.mean(-1,keepdim=True)
        var = x.var(-1,keepdim=True,unbiased=False)
        x = (x - mean)/torch.sqrt(var + self.eps)
        x = x * self.gamma + self.beta
        return x
    
class EncoderLayer(nn.Module):
    def __init__(self,dim,n_heads,hidden_size,dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(dim,n_heads,dropout)
        self.norm = layerNorm(dim)
        self.ffn = FeedForward(dim,hidden_size,dropout)
        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
        _x = x#自注意力层
        x = self.attention(x,x,x)
        x = self.norm(self.dropout(x) + _x)
        _x = x#位置前馈网络
        x = self.ffn(x)
        x = self.norm(self.dropout(x) + _x)##归一化+残差连接
        return x
    
class DecoderLayer(nn.Module):
    def __init__(self,dim,n_heads,hidden_size,dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(dim,n_heads,dropout)
        self.norm1 = layerNorm(dim)
        self.cross_attention = MultiHeadAttention(dim,n_heads,dropout)
        self.norm2 = layerNorm(dim)
        self.ffn = FeedForward(dim,hidden_size,dropout)
        self.norm3 = layerNorm(dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self,x,enc_output,src_mask,trg_mask):
        _x = x
        x =self.self_attention(x,x,x,trg_mask)
        x = self.norm1(self.dropout(x) + _x)
        
        _x = x
        x = self.cross_attention(x,enc_output,enc_output,src_mask)
        x = self.norm2(self.dropout(x) + _x)
        
        _x = x
        x = self.ffn(x)
        x = self.norm3(self.dropout(x) + _x)
        
        return x
      
        
class Transformer(nn.Module):
    def __init__(self,src_vocab_size,trg_vocab_size,dim=512,n_heads=8,n_layer=6,hidden_dim=2048,dropout=0.1,max_seq_len=5000):
        super().__init__()
        self.dim = dim
        self.src_embedding = nn.Embedding(src_vocab_size,dim)
        self.trg_embedding = nn.Embedding(trg_vocab_size,dim)
        self.pos_encoding = PositionalEncoding(dim,max_seq_len)
        self.dropout = nn.Dropout(dropout)
        
        self.encoder_layers = nn.ModuleList([EncoderLayer(dim,n_heads,hidden_dim,dropout) for _ in range(n_layer)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(dim,n_heads,hidden_dim,dropout) for _ in range(n_layer)])
        self.fc_out = nn.Linear(dim,trg_vocab_size)
        self._init_parameters()
        
    def _init_parameters(self):#参数初始化
        for p in self.parameters():
            if p.dim()>1:
                nn.init.xavier_uniform_(p)
                
    def create_masks(self,src,trg):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)# [batch_size,1,1,src_seq_len]
        trg_pad_mask = (trg != 0).unsqueeze(1).unsqueeze(2)# [batch_size,1,1,trg_seq_len]
        trg_len = trg.size(1)
        trg_sub_mask = torch.tril(torch.ones((trg_len,trg_len),device=trg.device)).bool()# [trg_seq_len,trg_seq_len]
        trg_mask = trg_pad_mask & trg_sub_mask# [batch_size,1,trg
        return src_mask,trg_mask
    
    def encode(self,src,src_mask):
        x = self.src_embedding(src)*math.sqrt(self.dim)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        for layer in self.encoder_layers:
            x = layer(x)#多层EncoderLayer
        return x
    
    def decode(self,trg,enc_output,src_mask,trg_mask):
        x = self.trg_embedding(trg)*math.sqrt(self.dim)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        for layer in self.decoder_layers:
            x = layer(x,enc_output,src_mask,trg_mask)#多层DecoderLayer
        return x
    
    def forward(self,src,trg):
        src_mask,trg_mask = self.create_masks(src,trg)
        enc_output = self.encode(src,src_mask)
        dec_output = self.decode(trg,enc_output,src_mask,trg_mask)
        output = self.fc_out(dec_output)
        return output

def main():
    src_vocab_size = 100
    tgt_vocab_size = 100
    dim = 512
    n_heads = 8
    n_layers = 6
    hidden_dim = 2048
    max_seq_len = 50
    dropout = 0.1
    
    # 实例化模型
    model = Transformer(
        src_vocab_size, 
        tgt_vocab_size, 
        dim, 
        n_heads=n_heads, 
        n_layer=n_layers, 
        hidden_dim=hidden_dim, 
        dropout=dropout,
        max_seq_len=max_seq_len
    )
    
    # 模拟输入数据
    batch_size = 2
    src_len = 10
    tgt_len = 12
    
    # 随机生成 src 和 tgt 序列 (假设 pad_token_id=0)
    # 确保没有 pad token 影响简单测试，或者手动插入
    src = torch.randint(1, src_vocab_size, (batch_size, src_len))
    tgt = torch.randint(1, tgt_vocab_size, (batch_size, tgt_len))
    
    # 前向传播
    output = model(src, tgt)
    
    print("Model Architecture:")
    # print(model)
    print("\nTest Input:")
    print(f"Source Shape: {src.shape}")
    print(f"Target Shape: {tgt.shape}")
    
    print("\nModel Output:")
    print(f"Output Shape: {output.shape}") # 预期 [batch_size, tgt_len, tgt_vocab_size]

if __name__ == "__main__":
    main()


Model Architecture:

Test Input:
Source Shape: torch.Size([2, 10])
Target Shape: torch.Size([2, 12])

Model Output:
Output Shape: torch.Size([2, 12, 100])
